# 02 · Main experiments

Raw, PCA-1, k-means LEACE, Metadata-LEACE and SAE-LEACE on every setting, the coverage sweeps, the geometry of the
confounder subspace, frozen transfer to the MTEB targets, and the ten-encoder study on the original corpora.
Results: `results/main/raw/`. Every step is cached: a finished file is reloaded, so the notebook can be run over
several sessions. Run on a GPU (Colab); the whole notebook takes about 30 hours of GPU time.

| Section | Feeds |
|---|---|
| 1. In-distribution comparison | Table 2, the extended in-distribution table, Figure 2 C, the references of Table 3 |
| 2. Rank and budget sweeps, mechanism | Figure 1, Table 2 and the extended in-distribution table (SCOTUS with E5-small), the reference check of Table 3 |
| 3. Geometry | Figure 2 A–B |
| 4. Frozen transfer | The frozen-transfer figure |
| 5. Ten-encoder study, original corpora | The ten-encoder table |

Seeds 0–2 or 0–4 as registered per setting (`code/corpora.py`); seeds 3–4 of the three-seed cells are added in
`04_extensions.ipynb`.

In [ ]:
# On Colab: mount Google Drive and move to your copy of the repository (see the README: "Running on Google Colab";
# set REPO if your copy is elsewhere). Skipped outside Colab.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/label-free-deconfounding-embeddings'
    %cd {REPO}
    %pip install -q -r requirements.txt
except ImportError:
    print('not on Colab: using the current working directory')

In [ ]:
import os, sys
if not os.path.isdir('code') and os.path.isdir(os.path.join('..', 'code')):
    os.chdir('..')                                   # opened from notebooks/
assert os.path.isdir('code') and os.path.isdir('data'), f'run from the repository root; cwd={os.getcwd()}'
sys.path.insert(0, 'code')

import torch                                         # first: some modules load diptest, whose macOS wheel bundles libomp
import paths

# QUICK = True runs every section end to end in minutes on a subset (two seeds, a 20-epoch SAE, a few settings),
# writing to results/*_quick/ so it never mixes with the real results. Its numbers are NOT the paper's.
QUICK = False
paths.configure(quick_run=QUICK)                     # before the experiment modules are imported
DEVICE = 'cuda' if torch.cuda.is_available() else None   # None: each step's own choice
print('cwd:', os.getcwd(), '| device:', DEVICE or 'auto', '| QUICK' if QUICK else '')

In [ ]:
import artifacts, experiments as E, ood
import revised as R
import alternatives as A

artifacts.set_results_dir(paths.MAIN)
E.set_embed_cache(paths.EMBED_CACHE)
ood.PROMPT_STYLE = 'parent'          # MTEB targets encoded plainly, as the parent paper did (Raw scores reproduce it)

E5_SETTINGS = ('semeval', 'swiltra', 'bills_tweets', 'scotus_oyez_lexis', 'cap', 'swiltra_de_it', 'swiltra_fr_it',
               'swiltra_de_fr_it', 'scotus_oyez_wiki', 'scotus_wiki_lexis', 'scotus_oyez_lexis_wiki')
MINI_SETTINGS = ('scotus_oyez_lexis', 'scotus_oyez_lexis_wiki', 'cap')        # MiniLM: the English settings
FIG1_SETTINGS = ('swiltra', 'swiltra_de_fr_it', 'scotus_oyez_lexis', 'scotus_oyez_lexis_wiki')
FIG2_ROWS = (('swiltra', 'e5-small'), ('swiltra_de_fr_it', 'e5-small'), ('scotus_oyez_lexis', 'e5-small'),
             ('scotus_oyez_lexis_wiki', 'e5-small'), ('scotus_oyez_lexis', 'mini'), ('scotus_oyez_lexis_wiki', 'mini'))
#: Frozen-transfer donors: the original five (`revised.OOD_MATRIX`) and the three-class ones of the frozen-transfer figure.
THREE_CLASS_DONORS = (('e5-small', 'scotus_oyez_lexis_wiki'), ('e5-small', 'swiltra_de_fr_it'),
                      ('mini', 'scotus_oyez_lexis_wiki'))
STUDY_CORPORA = ('semeval', 'swiltra', 'scotus_oyez_lexis', 'cap')          # the ten-encoder study's original corpora
#: The arms of the coverage runs that are also probed nonlinearly.
COVERAGE_NONLINEAR_ARMS = ('raw', 'pca_top1', 'pca_top2', 'pca_top3', 'pca_top4', 'pca_top8', 'pca_top16', 'pca_top32',
                           'pca_matched_var', 'kmeans_leace', 'sae_mi', 'sae_random', 'sae_recon',
                           'sae_recon_suppressed', 'metadata_leace')
SWEEP_SEEDS = R.SWEEP_SEEDS          # (0, 1, 2)
NONLINEAR = ('gb', 'mlp')
BUDGET_GRID = R.BUDGET_GRID          # N = 1 ... 64 SAE latents (Figure 1's curves)
ENCODER_STUDY = R.ENCODER_STUDY      # the ten encoders
OVERRIDES = {}
if QUICK:
    E5_SETTINGS, MINI_SETTINGS = ('swiltra_de_fr_it', 'cap'), ('scotus_oyez_lexis_wiki',)
    FIG1_SETTINGS, FIG2_ROWS = ('swiltra_de_fr_it',), (('swiltra_de_fr_it', 'e5-small'),)
    THREE_CLASS_DONORS, STUDY_CORPORA, ENCODER_STUDY = (('e5-small', 'swiltra_de_fr_it'),), ('swiltra',), ('mini', 'e5-small')
    SWEEP_SEEDS, NONLINEAR, BUDGET_GRID, OVERRIDES = (0, 1), (), (2, 8, 32), dict(n_epochs=20)
ID_SEEDS = SWEEP_SEEDS if QUICK else None          # None: each setting's registered seeds
OOD_SEEDS = SWEEP_SEEDS if QUICK else R.OOD_SEEDS   # the transfer donors' first three ID fits (0, 1, 2)
OOD_DONORS = tuple(d for d in R.OOD_MATRIX if not QUICK or d == ('e5-small', 'cap')) + THREE_CLASS_DONORS
R.MAP_DONORS.update(THREE_CLASS_DONORS)              # their run_id fits save frozen maps for the transfer study
print('recipe:', {**E.RECIPE, **OVERRIDES}, '| selection:', R.SELECTION)

## 1. In-distribution comparison

`revised.run_id` fits, per seed and outer split, PCA-1, Metadata-LEACE and SAE-LEACE (TopK SAE, MI-selected
top-32 latents against the silhouette-K k-means pseudo-groups, Directions-LEACE) and audits Raw and every map
(leakage: linear, GB and MLP probe AUC; utility: R@1/R@10 or topic purity). `revised.run_id_kmeans` adds k-means LEACE
on the same splits. The first three seeds of the transfer donors save their frozen maps under `embedding_cache/`.
CAP's nonlinear probes were added afterwards by `extend_id_nonlinear` (re-scored from the saved maps), as here.

In [ ]:
for key in E5_SETTINGS:
    if key == 'cap':
        R.run_id('cap', 'e5-small', seeds=ID_SEEDS, device=DEVICE, **OVERRIDES)
        if NONLINEAR:
            R.extend_id_nonlinear('cap', 'e5-small', probes=NONLINEAR, seeds=ID_SEEDS, device=DEVICE, **OVERRIDES)
    else:
        R.run_id(key, 'e5-small', seeds=ID_SEEDS, nonlinear=NONLINEAR, device=DEVICE, **OVERRIDES)
for key in MINI_SETTINGS:
    R.run_id(key, 'mini', seeds=ID_SEEDS, device=DEVICE, **OVERRIDES)

In [ ]:
for key in E5_SETTINGS:
    R.run_id_kmeans(key, 'e5-small', nonlinear=NONLINEAR, device=DEVICE)
for key in MINI_SETTINGS:
    if key != 'cap':                 # CAP/MiniLM has no k-means run
        R.run_id_kmeans(key, 'mini', nonlinear=(), device=DEVICE)

## 2. Rank and budget sweeps (the "coverage" runs) and the mechanism runs

`alternatives.run_alternatives` refits on seeds 0–2: PCA at ranks 1–64, displacement-matched PCA, k-means LEACE,
SAE-LEACE with MI-selected and random latents along a budget grid of N latents, and the reconstruction controls.
SemEval and CAP use the first grid (N = 4–32, every arm probed nonlinearly); the four Figure 1 settings use the full
grid (N = 1–64). `run_kmeans_count` adds k-means LEACE at K = 2–8 and K = C. `revised.run_mechanism` (Activation-LEACE,
orthogonal removal, Directions-LEACE on the same latents) is read by the reference check of the Table 3 runs.

In [ ]:
for key in [k for k in ('semeval', 'cap') if not QUICK]:
    A.run_alternatives(key, 'e5-small', ranks=A.RANKS, seeds=SWEEP_SEEDS, budget_grid=(4, 8, 16, 32),
                       nonlinear=NONLINEAR, nonlinear_arms=None, selectors=True, device=DEVICE, **OVERRIDES)
for key in FIG1_SETTINGS:
    A.run_alternatives(key, 'e5-small', ranks=A.RANKS, seeds=SWEEP_SEEDS, budget_grid=BUDGET_GRID,
                       nonlinear=NONLINEAR, nonlinear_arms=COVERAGE_NONLINEAR_ARMS, selectors=False,
                       device=DEVICE, **OVERRIDES)
    A.run_kmeans_count(key, 'e5-small', seeds=SWEEP_SEEDS, nonlinear=NONLINEAR)

In [ ]:
if not QUICK:
    R.run_mechanism('semeval', seeds=SWEEP_SEEDS, device=DEVICE, **OVERRIDES)
    R.extend_mechanism_nonlinear('semeval', probes=NONLINEAR, device=DEVICE, **OVERRIDES)
for key in ('swiltra_de_fr_it', 'scotus_oyez_lexis_wiki'):
    R.run_mechanism(key, seeds=SWEEP_SEEDS, nonlinear=NONLINEAR, device=DEVICE, **OVERRIDES)

## 3. Geometry (post hoc)

Where the confounder sits: the share of the supervised confounder subspace captured by the leading principal
components, and the adjusted Rand index of the silhouette-K k-means groups with the classes. Uses the labels: a
diagnostic, not part of any method.

In [ ]:
for key, enc in FIG2_ROWS:
    A.run_geometry(key, enc, seeds=SWEEP_SEEDS, device=DEVICE)

## 4. Frozen transfer to the MTEB targets

Each donor's maps (PCA-1, Metadata-LEACE, SAE-LEACE, k-means LEACE, PCA at ranks 1–32 and displacement-matched PCA)
are fitted on the donor's in-distribution training split (seeds 0–2), frozen, and applied to five legal retrieval
tasks and STSBenchmark.

In [ ]:
for enc, donor in OOD_DONORS:
    R.run_ood(enc, donor, seeds=OOD_SEEDS, device=DEVICE)
ORIGINAL = [d for d in OOD_DONORS if d in R.OOD_MATRIX]
A.run_ood_kmeans(matrix=ORIGINAL, seeds=OOD_SEEDS, generous=True, device=DEVICE)              # adds K = 8
A.run_ood_kmeans(matrix=list(THREE_CLASS_DONORS), seeds=OOD_SEEDS, generous=False, device=DEVICE)
A.run_ood_ranks(matrix=ORIGINAL, ranks=A.OOD_RANKS, matched_var=True, seeds=OOD_SEEDS, device=DEVICE)

## 5. Ten-encoder study on the original corpora (the ten-encoder table)

Every eraser refitted in each encoder's embedding space (seeds 0–2): `run_encoder_cell` (Raw, PCA ranks 1–32,
Metadata-LEACE, SAE-LEACE), `run_generous_k` (k-means LEACE at the silhouette K, K = C and K = 8) and
`run_geometry`. The three-class settings and Fedlex follow in notebooks 04 and 03.

In [ ]:
for enc in ENCODER_STUDY:
    for corpus in STUDY_CORPORA:
        R.run_encoder_cell(corpus, enc, seeds=R.ENCODER_SEEDS if not QUICK else SWEEP_SEEDS,
                           pca_ranks=R.ENCODER_PCA_GRID, device=DEVICE, verbose=False)
        A.run_generous_k(corpus, enc, seeds=R.ENCODER_SEEDS if not QUICK else SWEEP_SEEDS, nonlinear=NONLINEAR,
                         device=DEVICE)
        A.run_geometry(corpus, enc, seeds=R.ENCODER_SEEDS if not QUICK else SWEEP_SEEDS, device=DEVICE)